In [ ]:
"""
Britain Bicycle Accident Severity Prediction (Paper Reproduction)
==================================================================
Based on: Random Survival Forest (RSF) + Random Parameters Logit Model (RPLM)
Data: STATS19 (DfT, UK) — Bicycle accidents 2016–2018
Reference: Paper on forecasting bicycle accident severity in Britain (2016–2018)
"""
import os, sys, warnings, json, time, random, pickle
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

In [ ]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

In [ ]:
BASE = os.path.normpath(os.path.dirname(os.path.abspath(__file__)))
DATA_DIR = os.path.join(BASE, 'data', 'britain-dataset')
OUTPUT_DIR = os.path.join(BASE, 'outputs', 'britain_paper_reproduction')
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
COLLISION_FILE = os.path.join(DATA_DIR, 'dft-road-casualty-statistics-collision-1979-latest-published-year.csv')
VEHICLE_FILE = os.path.join(DATA_DIR, 'dft-road-casualty-statistics-vehicle-1979-latest-published-year.csv')
CASUALTY_FILE = os.path.join(DATA_DIR, 'dft-road-casualty-statistics-casualty-1979-latest-published-year.csv')

In [ ]:
print("=" * 70)
print("BRITAIN BICYCLE ACCIDENT SEVERITY PREDICTION (Paper Reproduction)")
print("=" * 70)

In [ ]:
# ============================================================
# 1. DATA LOADING — Identify bicycle collisions 2016–2018
# ============================================================
print("\n[1] Identifying bicycle-involved collisions (2016–2018)...")
t0 = time.time()

In [ ]:
# vehicle_type=1 is the code for pedal cycle in the STATS19 vehicle table
VCOLS = ['collision_index', 'collision_year', 'vehicle_type', 'vehicle_reference',
         'sex_of_driver', 'age_of_driver', 'age_band_of_driver',
         'vehicle_manoeuvre', 'first_point_of_impact', 'journey_purpose_of_driver',
         'age_of_vehicle', 'propulsion_code', 'engine_capacity_cc']

In [ ]:
reader = pd.read_csv(VEHICLE_FILE, usecols=VCOLS, low_memory=False, chunksize=500000)

In [ ]:
bike_vehicle_rows = []
all_vehicle_rows_for_bike_crashes = []
bike_collision_indices = set()

In [ ]:
for i, chunk in enumerate(reader):
    yr_mask = chunk['collision_year'].between(2016, 2018)
    if not yr_mask.any():
        continue
    chunk_yr = chunk[yr_mask].copy()
    is_bike = chunk_yr['vehicle_type'] == 1
    bike_rows = chunk_yr[is_bike]
    if len(bike_rows) > 0:
        bike_vehicle_rows.append(bike_rows)
        bike_collision_indices.update(bike_rows['collision_index'].unique())

In [ ]:
print(f"  Found {len(bike_collision_indices):,} unique bicycle-involved collisions")
print(f"  Time: {time.time()-t0:.1f}s")

In [ ]:
# ============================================================
# 2. LOAD COLLISION DATA for these indices
# ============================================================
print("\n[2] Loading collision data...")
t0 = time.time()

In [ ]:
CCOLS = ['collision_index', 'collision_year', 'collision_severity',
         'number_of_vehicles', 'number_of_casualties', 'date', 'day_of_week', 'time',
         'road_type', 'speed_limit', 'junction_detail', 'junction_control',
         'light_conditions', 'weather_conditions', 'road_surface_conditions',
         'urban_or_rural_area', 'first_road_class', 'second_road_class',
         'pedestrian_crossing', 'longitude', 'latitude']

In [ ]:
collision_chunks = []
for chunk in pd.read_csv(COLLISION_FILE, usecols=CCOLS, low_memory=False, chunksize=500000):
    matched = chunk[chunk['collision_index'].isin(bike_collision_indices)]
    if len(matched) > 0:
        collision_chunks.append(matched)
collision_df = pd.concat(collision_chunks, ignore_index=True)
print(f"  Loaded {len(collision_df):,} collision records")
print(f"  Time: {time.time()-t0:.1f}s")

In [ ]:
# ============================================================
# 3. LOAD & FILTER VEHICLE DATA
# ============================================================
print("\n[3] Loading vehicle data for bicycle collisions...")
t0 = time.time()

In [ ]:
reader = pd.read_csv(VEHICLE_FILE, usecols=VCOLS, low_memory=False, chunksize=500000)
vehicle_chunks = []
for chunk in reader:
    matched = chunk[chunk['collision_index'].isin(bike_collision_indices)]
    if len(matched) > 0:
        vehicle_chunks.append(matched)
vehicle_df = pd.concat(vehicle_chunks, ignore_index=True)
print(f"  Loaded {len(vehicle_df):,} vehicle records")
print(f"  Time: {time.time()-t0:.1f}s")

In [ ]:
# ============================================================
# 4. FILTER: Exactly 1 bicycle + at most 1 other vehicle
# ============================================================
print("\n[4] Filtering: exactly 1 bicycle + at most 1 other vehicle...")
t0 = time.time()

In [ ]:
# Count vehicles per collision
vehicle_counts = vehicle_df.groupby('collision_index')['vehicle_type'].agg([
    ('n_vehicles', 'count'),
    ('n_bikes', lambda x: (x == 1).sum()),
    ('n_other', lambda x: (x != 1).sum())
]).reset_index()

In [ ]:
valid = vehicle_counts[(vehicle_counts['n_bikes'] == 1) & (vehicle_counts['n_other'] <= 1)]
valid_indices = set(valid['collision_index'])
print(f"  Valid collisions (1 bike, <=1 other): {len(valid_indices):,}")

In [ ]:
# Filter both datasets
collision_df = collision_df[collision_df['collision_index'].isin(valid_indices)].copy()
vehicle_df = vehicle_df[vehicle_df['collision_index'].isin(valid_indices)].copy()

In [ ]:
# Separate bicycle vehicle row and other vehicle row per collision
bike_vehicles = vehicle_df[vehicle_df['vehicle_type'] == 1].copy()
other_vehicles = vehicle_df[vehicle_df['vehicle_type'] != 1].copy()

In [ ]:
# Rename other vehicle columns to avoid collision
bike_vehicles = bike_vehicles.add_prefix('bike_')
other_vehicles = other_vehicles.add_prefix('other_')

In [ ]:
bike_vehicles.rename(columns={'bike_collision_index': 'collision_index'}, inplace=True)
other_vehicles.rename(columns={'other_collision_index': 'collision_index'}, inplace=True)

In [ ]:
# Merge collision + bike + other vehicle
data = collision_df.merge(bike_vehicles, on='collision_index', how='left')
data = data.merge(other_vehicles, on='collision_index', how='left')
print(f"  Merged: {len(data):,} records")
print(f"  Time: {time.time()-t0:.1f}s")

In [ ]:
# ============================================================
# 5. LOAD CASUALTY DATA
# ============================================================
print("\n[5] Loading casualty data...")
t0 = time.time()

In [ ]:
CASCOLS = ['collision_index', 'vehicle_reference', 'casualty_reference',
           'casualty_class', 'sex_of_casualty', 'age_of_casualty',
           'age_band_of_casualty', 'casualty_severity', 'casualty_type']

In [ ]:
casualty_chunks = []
for chunk in pd.read_csv(CASUALTY_FILE, usecols=CASCOLS, low_memory=False, chunksize=500000):
    matched = chunk[chunk['collision_index'].isin(valid_indices)]
    if len(matched) > 0:
        casualty_chunks.append(matched)
casualty_df = pd.concat(casualty_chunks, ignore_index=True)

In [ ]:
# Aggregate casualty info per collision
casualty_agg = casualty_df.groupby('collision_index').agg(
    n_casualties_actual=('casualty_reference', 'count'),
    n_fatal_casualties=('casualty_severity', lambda x: (x == 1).sum()),
    n_serious_casualties=('casualty_severity', lambda x: (x == 2).sum()),
    mean_casualty_age=('age_of_casualty', 'mean'),
    pct_male_casualties=('sex_of_casualty', lambda x: (x == 1).mean()),
).reset_index()

In [ ]:
data = data.merge(casualty_agg, on='collision_index', how='left')
print(f"  Casualty data merged: {len(data):,} records")
print(f"  Time: {time.time()-t0:.1f}s")

In [ ]:
# ============================================================
# 6. TARGET VARIABLE
# ============================================================
print("\n[6] Defining target variable (severity)...")

In [ ]:
# collision_severity: 1=Fatal, 2=Serious, 3=Slight
severity_map = {1: 2, 2: 1, 3: 0}  # map to 0=Slight, 1=Serious, 2=Fatal for ordered classes
data['severity_class'] = data['collision_severity'].map(severity_map)

In [ ]:
sev_counts = data['collision_severity'].value_counts().sort_index()
sev_total = len(data)
sev_labels = {1: 'Fatal', 2: 'Serious', 3: 'Slight'}
for sev in sorted(sev_counts.index):
    print(f"  {sev_labels[sev]:8s}: {sev_counts[sev]:>6,} ({sev_counts[sev]/sev_total*100:5.2f}%)")
print(f"  Total: {sev_total:,}")

In [ ]:
# ============================================================
# 7. FEATURE ENGINEERING
# ============================================================
print("\n[7] Feature engineering (dummy variables)...")
t0 = time.time()

In [ ]:
# --- Collision-level features ---
data['speed_limit_ge50'] = (data['speed_limit'] >= 50).astype(int)
data['is_dark'] = data['light_conditions'].isin([4, 5, 6, 7]).astype(int)
data['is_urban'] = (data['urban_or_rural_area'] == 1).astype(int)
data['is_rural'] = (data['urban_or_rural_area'] == 2).astype(int)
data['is_wet_road'] = (data['road_surface_conditions'] == 2).astype(int)
data['is_rain'] = (data['weather_conditions'].isin([2, 5])).astype(int)
data['is_weekend'] = data['day_of_week'].isin([1, 7]).astype(int)
data['is_single_carriageway'] = (data['road_type'] == 6).astype(int)
data['is_dual_carriageway'] = (data['road_type'] == 3).astype(int)
data['is_roundabout'] = (data['road_type'] == 1).astype(int)
data['is_junction'] = (data['junction_detail'] > 0).astype(int)

In [ ]:
# --- Bicycle-level features ---
data['bike_male'] = (data['bike_sex_of_driver'] == 1).astype(int)
data['bike_female'] = (data['bike_sex_of_driver'] == 2).astype(int)

In [ ]:
# Cyclist age groups (paper emphasis on age >= 75)
data['bike_age_unknown'] = (data['bike_age_of_driver'] < 0).astype(int)
data['bike_age_0_15'] = (data['bike_age_of_driver'].between(0, 15)).astype(int)
data['bike_age_16_24'] = (data['bike_age_of_driver'].between(16, 24)).astype(int)
data['bike_age_25_34'] = (data['bike_age_of_driver'].between(25, 34)).astype(int)
data['bike_age_35_44'] = (data['bike_age_of_driver'].between(35, 44)).astype(int)
data['bike_age_45_54'] = (data['bike_age_of_driver'].between(45, 54)).astype(int)
data['bike_age_55_64'] = (data['bike_age_of_driver'].between(55, 64)).astype(int)
data['bike_age_65_74'] = (data['bike_age_of_driver'].between(65, 74)).astype(int)
data['bike_age_75plus'] = (data['bike_age_of_driver'] >= 75).astype(int)

In [ ]:
# --- Other vehicle driver features ---
data['other_male'] = (data['other_sex_of_driver'] == 1).astype(int)
data['other_female'] = (data['other_sex_of_driver'] == 2).astype(int)
data['other_age_unknown'] = (data['other_age_of_driver'] < 0).astype(int)
data['other_age_0_15'] = (data['other_age_of_driver'].between(0, 15)).astype(int)
data['other_age_16_24'] = (data['other_age_of_driver'].between(16, 24)).astype(int)
data['other_age_25_34'] = (data['other_age_of_driver'].between(25, 34)).astype(int)
data['other_age_35_44'] = (data['other_age_of_driver'].between(35, 44)).astype(int)
data['other_age_45_54'] = (data['other_age_of_driver'].between(45, 54)).astype(int)
data['other_age_55_64'] = (data['other_age_of_driver'].between(55, 64)).astype(int)
data['other_age_65_74'] = (data['other_age_of_driver'].between(65, 74)).astype(int)
data['other_age_75plus'] = (data['other_age_of_driver'] >= 75).astype(int)

In [ ]:
# Other vehicle type
data['other_is_car'] = (data['other_vehicle_type'] == 9).astype(int)
data['other_is_motorcycle'] = (data['other_vehicle_type'].isin([2, 3, 4, 5, 23, 97, 109])).astype(int)
data['other_is_lgv'] = (data['other_vehicle_type'].isin([19, 20, 21])).astype(int)
data['other_is_hgv'] = (data['other_vehicle_type'] == 21).astype(int)
data['other_is_bus'] = (data['other_vehicle_type'].isin([10, 11])).astype(int)
data['other_is_taxi'] = (data['other_vehicle_type'] == 8).astype(int)

In [ ]:
# Other vehicle manoeuvre (paper emphasizes this as important)
data['other_turning'] = data['other_vehicle_manoeuvre'].isin([17, 18, 19, 20, 21, 22, 23, 24, 25]).astype(int)
data['other_overtaking'] = data['other_vehicle_manoeuvre'].isin([12, 13, 14]).astype(int)
data['other_reversing'] = (data['other_vehicle_manoeuvre'] == 9).astype(int)
data['other_parked'] = (data['other_vehicle_manoeuvre'] == 7).astype(int)

In [ ]:
# Whether there is a second vehicle
data['has_other_vehicle'] = data['other_vehicle_type'].notna().astype(int)

In [ ]:
# --- Combined features ---
# Speed limit >= 50 mph (paper: increases fatal probability by 36.63%)
data['speed_ge50_and_other'] = (data['speed_limit_ge50'] & data['has_other_vehicle'] == 1).astype(int)

In [ ]:
print(f"  Total features created")
print(f"  Time: {time.time()-t0:.1f}s")

In [ ]:
# ============================================================
# 8. PREPARE MODEL MATRIX
# ============================================================
print("\n[8] Preparing model matrix...")

In [ ]:
FEATURE_COLS = [
    'speed_limit', 'speed_limit_ge50',
    'is_dark', 'is_urban', 'is_rural', 'is_wet_road', 'is_rain',
    'is_weekend', 'is_single_carriageway', 'is_dual_carriageway',
    'is_roundabout', 'is_junction',
    'bike_male', 'bike_female',
    'bike_age_0_15', 'bike_age_16_24', 'bike_age_25_34',
    'bike_age_35_44', 'bike_age_45_54', 'bike_age_55_64',
    'bike_age_65_74', 'bike_age_75plus', 'bike_age_unknown',
    'other_male', 'other_female',
    'other_age_0_15', 'other_age_16_24', 'other_age_25_34',
    'other_age_35_44', 'other_age_45_54', 'other_age_55_64',
    'other_age_65_74', 'other_age_75plus', 'other_age_unknown',
    'other_is_car', 'other_is_motorcycle', 'other_is_lgv',
    'other_is_hgv', 'other_is_bus', 'other_is_taxi',
    'other_turning', 'other_overtaking', 'other_reversing', 'other_parked',
    'has_other_vehicle',
]

In [ ]:
# Drop rows with missing critical features
data = data.dropna(subset=['severity_class', 'speed_limit']).copy()
for col in FEATURE_COLS:
    if col not in data.columns:
        data[col] = 0
    data[col] = data[col].fillna(0)

In [ ]:
X = data[FEATURE_COLS].values.astype(float)
y = data['severity_class'].values.astype(int)  # 0=Slight, 1=Serious, 2=Fatal

In [ ]:
# Also keep 3-class one-hot for multi-class evaluation
from sklearn.preprocessing import label_binarize
y_bin = label_binarize(y, classes=[0, 1, 2])

In [ ]:
print(f"  Features: {X.shape[1]}")
print(f"  Samples:  {X.shape[0]:,}")
print(f"  Class distribution: Slight={(y==0).sum():,}, Serious={(y==1).sum():,}, Fatal={(y==2).sum():,}")

In [ ]:
# ============================================================
# 9. TRAIN/TEST SPLIT (70/30)
# ============================================================
print("\n[9] Train/test split (70/30)...")
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_SEED, stratify=y
)
print(f"  Train: {len(X_train):,} | Test: {len(X_test):,}")

In [ ]:
# Standardize numerical features
from sklearn.preprocessing import StandardScaler
num_cols_idx = [0]  # speed_limit column index
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[:, num_cols_idx] = scaler.fit_transform(X_train[:, num_cols_idx].reshape(-1, 1))
X_test_scaled[:, num_cols_idx] = scaler.transform(X_test[:, num_cols_idx].reshape(-1, 1))

In [ ]:
# ============================================================
# EVALUATION METRICS (Paper: F-measure, G-mean, FP rate)
# ============================================================
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [ ]:
def compute_metrics(y_true, y_pred, y_prob=None, model_name='Model'):
    """Compute paper metrics: F-measure, G-mean, FP rate per class + macro."""
    n_classes = 3
    metrics = {'model': model_name}

    # Per-class metrics
    for c in range(n_classes):
        mask_c = (y_true == c)
        tp = ((y_pred == c) & mask_c).sum()
        fp = ((y_pred == c) & ~mask_c).sum()
        fn = (~(y_pred == c) & mask_c).sum()
        tn = (~(y_pred == c) & ~mask_c).sum()

        prec_c = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec_c = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1_c = 2 * prec_c * rec_c / (prec_c + rec_c) if (prec_c + rec_c) > 0 else 0.0
        fp_rate_c = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        spec_c = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        gmean_c = np.sqrt(rec_c * spec_c) if (rec_c * spec_c) >= 0 else 0.0

        label = ['Slight', 'Serious', 'Fatal'][c]
        metrics[f'{label}_Precision'] = round(prec_c, 4)
        metrics[f'{label}_Recall'] = round(rec_c, 4)
        metrics[f'{label}_F1'] = round(f1_c, 4)
        metrics[f'{label}_FPrate'] = round(fp_rate_c, 4)
        metrics[f'{label}_Gmean'] = round(gmean_c, 4)

    # Macro averages
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    macro_prec = precision_score(y_true, y_pred, average='macro')
    macro_rec = recall_score(y_true, y_pred, average='macro')
    acc = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)

    metrics['Accuracy'] = round(acc, 4)
    metrics['Macro_Precision'] = round(macro_prec, 4)
    metrics['Macro_Recall'] = round(macro_rec, 4)
    metrics['Macro_F1'] = round(macro_f1, 4)

    print(f"\n  {model_name}:")
    print(f"    Accuracy:      {acc:.4f}")
    print(f"    Macro-Precision: {macro_prec:.4f}")
    print(f"    Macro-Recall:    {macro_rec:.4f}")
    print(f"    Macro-F1:        {macro_f1:.4f}")
    print(f"    Confusion Matrix:")
    print(f"      {cm[0]}  (True Slight)")
    print(f"      {cm[1]}  (True Serious)")
    print(f"      {cm[2]}  (True Fatal)")
    for c in range(n_classes):
        label = ['Slight', 'Serious', 'Fatal'][c]
        print(f"    {label}: F1={metrics[f'{label}_F1']:.4f}, G-mean={metrics[f'{label}_Gmean']:.4f}, "
              f"FPrate={metrics[f'{label}_FPrate']:.4f}")

    return metrics

In [ ]:
# ============================================================
# 10. MODEL 1: RANDOM SURVIVAL FOREST (RSF)
# ============================================================
print("\n" + "=" * 70)
print("MODEL 1: RANDOM SURVIVAL FOREST (RSF)")
print("=" * 70)
print("  Paper: 68 trees, depth=4, Gini reduction, if-then rules")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rsf_params = {
    'n_estimators': 68,
    'max_depth': 4,
    'criterion': 'gini',
    'min_samples_split': 20,
    'min_samples_leaf': 10,
    'max_features': 'sqrt',
    'random_state': RANDOM_SEED,
    'n_jobs': -1,
    'class_weight': 'balanced',
}
print(f"\n  Hyperparameters: {rsf_params}")

In [ ]:
t0 = time.time()
rsf = RandomForestClassifier(**rsf_params)
rsf.fit(X_train, y_train)
rsf_train_time = time.time() - t0

In [ ]:
# Predict
y_pred_rsf = rsf.predict(X_test)
y_prob_rsf = rsf.predict_proba(X_test)

In [ ]:
# RandomForestClassifier returns (n_samples, n_classes) for multi-class
if isinstance(y_prob_rsf, list):
    y_prob_rsf_3c = np.column_stack([p[:, 1] if len(p.shape) > 1 and p.shape[1] >= 2 else p for p in y_prob_rsf])
else:
    y_prob_rsf_3c = np.array(y_prob_rsf)

In [ ]:
rsf_metrics = compute_metrics(y_test, y_pred_rsf, y_prob_rsf_3c, 'RSF')
rsf_metrics['train_time_s'] = round(rsf_train_time, 2)

In [ ]:
# Variable Importance
feature_importance = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': rsf.feature_importances_
}).sort_values('importance', ascending=False)
print(f"\n  Top 10 Variable Importance:")
for i, row in feature_importance.head(10).iterrows():
    print(f"    {row['feature']:25s}: {row['importance']:.4f}")

In [ ]:
feature_importance.to_csv(os.path.join(OUTPUT_DIR, 'rsf_variable_importance.csv'), index=False)

In [ ]:
# Extract if-then rules from trees (paper: 1,038 rules)
print(f"\n  Extracting if-then rules from trees...")
rules_counter = [0]
all_rules = []

In [ ]:
def _extract_rules(tree_, node_id, conditions, rules_counter, all_rules, tree_idx, feat_names):
    children_left = tree_.children_left
    children_right = tree_.children_right
    feature_idx = tree_.feature
    threshold = tree_.threshold
    values = tree_.value

    if children_left[node_id] == -1 and children_right[node_id] == -1:
        # Leaf node
        class_counts = values[node_id][0]
        majority_class = np.argmax(class_counts)
        if len(conditions) > 0:
            rules_counter[0] += 1
            rule = {
                'tree': tree_idx,
                'conditions': conditions.copy(),
                'predicted_class': int(majority_class),
                'samples': int(class_counts.sum()),
                'purity': float(class_counts[majority_class] / class_counts.sum())
            }
            all_rules.append(rule)
        return

    feat_idx = feature_idx[node_id]
    feat_name = feat_names[feat_idx] if feat_idx >= 0 and feat_idx < len(feat_names) else f'feat_{feat_idx}'
    thr = threshold[node_id]

    # Left: <= threshold
    cond_left = conditions + [(feat_name, '<=', thr)]
    _extract_rules(tree_, children_left[node_id], cond_left, rules_counter, all_rules, tree_idx, feat_names)

    # Right: > threshold
    cond_right = conditions + [(feat_name, '>', thr)]
    _extract_rules(tree_, children_right[node_id], cond_right, rules_counter, all_rules, tree_idx, feat_names)

In [ ]:
for tree_idx, tree in enumerate(rsf.estimators_):
    _extract_rules(tree.tree_, 0, [], rules_counter, all_rules, tree_idx, FEATURE_COLS)

In [ ]:
rules_count = rules_counter[0]

In [ ]:
print(f"  Extracted {rules_count:,} if-then rules (paper: 1,038)")

In [ ]:
# Save rules (top 50 with highest purity)
rules_df = pd.DataFrame(all_rules)
if len(rules_df) > 0:
    rules_df['rule_str'] = rules_df['conditions'].apply(
        lambda conds: ' AND '.join([f'{f} {op} {t:.1f}' for f, op, t in conds])
    )
    top_rules = rules_df.sort_values('purity', ascending=False).head(50)
    top_rules.to_csv(os.path.join(OUTPUT_DIR, 'rsf_top50_rules.csv'), index=False)
    print(f"  Top 50 rules saved to {OUTPUT_DIR}")

In [ ]:
# ============================================================
# 11. MODEL 2: RANDOM PARAMETERS LOGIT MODEL (RPLM)
# ============================================================
print("\n" + "=" * 70)
print("MODEL 2: RANDOM PARAMETERS LOGIT MODEL (RPLM)")
print("=" * 70)
print("  Mixed Logit with simulation-based maximum likelihood")
print("  Random parameters: normal distribution (Halton simulation)")
print("  Paper: cyclist age>=75 + male cyclist + driver 55-64")

In [ ]:
import numba
from numba import njit, prange
from scipy.optimize import minimize
from scipy.stats import norm

In [ ]:
# ---- Feature Selection ----
# Must include the 3 random-parameter features + top non-random features
FORCED_RANDOM_FEATURES = ['bike_age_75plus', 'bike_male', 'other_age_55_64']
random_feat_idx = [FEATURE_COLS.index(f) for f in FORCED_RANDOM_FEATURES]

In [ ]:
# Pick top non-random features: exclude the 3 random features and take next best
non_random_importance = feature_importance[~feature_importance['feature'].isin(FORCED_RANDOM_FEATURES)]
top_k_nonrandom = 12
top_nonrandom = non_random_importance.head(top_k_nonrandom)['feature'].tolist()
nonrandom_feat_idx = [FEATURE_COLS.index(f) for f in top_nonrandom]

In [ ]:
# Final feature list: non-random + random (random at the end for easy indexing)
all_feat_names = top_nonrandom + FORCED_RANDOM_FEATURES
all_feat_idx = nonrandom_feat_idx + random_feat_idx
n_fixed = len(nonrandom_feat_idx)
n_rand_features = len(random_feat_idx)
n_classes = 3

In [ ]:
print(f"\n  Features: {len(all_feat_names)} ({n_fixed} fixed + {n_rand_features} random)")
for f in all_feat_names:
    tag = " [RANDOM]" if f in FORCED_RANDOM_FEATURES else ""
    print(f"    - {f}{tag}")

In [ ]:
# ---- Random Coefficients Mapping (numba-compatible 2D array) ----
random_coeff_pairs = [
    (n_fixed + 0, 1),  # bike_age_75plus -> Serious
    (n_fixed + 0, 2),  # bike_age_75plus -> Fatal
    (n_fixed + 1, 1),  # bike_male -> Serious
    (n_fixed + 1, 2),  # bike_male -> Fatal
    (n_fixed + 2, 1),  # other_age_55_64 -> Serious
]
random_coeff_map = np.array(random_coeff_pairs, dtype=np.int64)
n_rand_coeffs = len(random_coeff_pairs)
print(f"  Random coefficients: {n_rand_coeffs}")
for (fi, cls) in random_coeff_pairs:
    print(f"    {all_feat_names[fi]:25s} -> class {cls}")

In [ ]:
# ---- Prepare Data ----
N_RPLM_TRAIN = 15000
rplm_idx = np.random.RandomState(RANDOM_SEED).permutation(len(X_train))[:N_RPLM_TRAIN]
X_tr = X_train[rplm_idx][:, all_feat_idx].astype(np.float64)
y_tr = y_train[rplm_idx].astype(np.int64)
X_te = X_test[:, all_feat_idx].astype(np.float64)
y_te = y_test

In [ ]:
n_fixed_params = n_fixed * (n_classes - 1)  # 12 * 2 = 24
n_total = n_fixed_params + n_rand_coeffs * 2  # means + stds
N_HALTON = 50

In [ ]:
# Pre-compute Halton draws
halton_draws = np.zeros((N_HALTON, n_rand_coeffs))
for d in range(n_rand_coeffs):
    for i in range(N_HALTON):
        base = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29][d % 10]
        vdc = 0.0
        denom = 1.0
        idx = i + 1
        while idx > 0:
            denom *= base
            idx, rem = divmod(idx, base)
            vdc += rem / denom
        halton_draws[i, d] = norm.ppf(vdc)

In [ ]:
print(f"\n  Training: {N_RPLM_TRAIN} samples, {n_total} params, {N_HALTON} Halton draws")

In [ ]:
# ---- Class Weights (inverse frequency) ----
class_counts = np.array([(y_tr == c).sum() for c in range(n_classes)], dtype=np.float64)
class_weights = class_counts.sum() / (n_classes * class_counts)
sample_weights = np.array([class_weights[yi] for yi in y_tr], dtype=np.float64)
print(f"  Class weights: Slight={class_weights[0]:.2f}, Serious={class_weights[1]:.2f}, Fatal={class_weights[2]:.2f}")

In [ ]:
# ---- Numba-accelerated Negative Log-Likelihood (weighted) ----
@njit
def _mixed_logit_nll(beta, X, y, sw, halton, n_fixed, n_fixed_params, n_rand_coeffs,
                     n_classes, random_coeff_map, n_halton, n_samples):
    """Numba-optimized weighted mixed logit negative log-likelihood."""
    beta_fixed = beta[:n_fixed_params]
    beta_rand_mean = beta[n_fixed_params:n_fixed_params + n_rand_coeffs]
    beta_rand_std_arr = beta[n_fixed_params + n_rand_coeffs:]
    for rc in range(n_rand_coeffs):
        if beta_rand_std_arr[rc] < 0.05:
            beta_rand_std_arr[rc] = 0.05

    total_ll = 0.0
    for s in range(n_samples):
        probs = np.zeros(n_classes)
        for r in range(n_halton):
            U = np.zeros(n_classes)
            pi = 0
            for c in range(1, n_classes):
                for fi in range(n_fixed):
                    U[c] += X[s, fi] * beta_fixed[pi]
                    pi += 1
            for rc in range(n_rand_coeffs):
                fi_rc = random_coeff_map[rc, 0]
                class_rc = random_coeff_map[rc, 1]
                draw = halton[r, rc]
                rb = beta_rand_mean[rc] + beta_rand_std_arr[rc] * draw
                U[class_rc] += X[s, fi_rc] * rb
            max_u = U[0]
            for c in range(1, n_classes):
                if U[c] > max_u:
                    max_u = U[c]
            exp_sum = 0.0
            for c in range(n_classes):
                exp_sum += np.exp(U[c] - max_u)
            for c in range(n_classes):
                probs[c] += np.exp(U[c] - max_u) / exp_sum
        probs /= n_halton
        for c in range(n_classes):
            if probs[c] < 1e-15:
                probs[c] = 1e-15
        total_ll += sw[s] * np.log(probs[y[s]])
    return -total_ll

In [ ]:
# ---- Warm-start with MNLogit ----
from sklearn.linear_model import LogisticRegression
warm_logit = LogisticRegression(solver='lbfgs', max_iter=500, C=1e6, random_state=RANDOM_SEED)
warm_logit.fit(X_tr[:, :n_fixed], y_tr)
warm_coef = warm_logit.coef_  # (n_classes, n_fixed)
warm_intercept = warm_logit.intercept_  # (n_classes,)

In [ ]:
beta_init = np.zeros(n_total)
pi = 0
for c in range(1, n_classes):
    for fi in range(n_fixed):
        beta_init[pi] = warm_coef[c, fi] - warm_coef[0, fi]
        pi += 1
for rc in range(n_rand_coeffs):
    beta_init[n_fixed_params + rc] = 0.01
    beta_init[n_fixed_params + n_rand_coeffs + rc] = 0.5

In [ ]:
# ---- Optimization ----
t0 = time.time()
n_samples = X_tr.shape[0]

In [ ]:
bounds = [(None, None)] * n_fixed_params
bounds += [(None, None)] * n_rand_coeffs
bounds += [(0.01, None)] * n_rand_coeffs

In [ ]:
result = minimize(
    _mixed_logit_nll, beta_init,
    args=(X_tr, y_tr, sample_weights, halton_draws, n_fixed, n_fixed_params, n_rand_coeffs,
          n_classes, random_coeff_map, N_HALTON, n_samples),
    method='L-BFGS-B',
    bounds=bounds,
    options={'maxiter': 100, 'ftol': 1e-4, 'gtol': 1e-4}
)
rplm_train_time = time.time() - t0

In [ ]:
beta_opt = result.x
print(f"\n  Optimization: {result.nit} iterations, converged={result.success}")
print(f"  Training time: {rplm_train_time:.1f}s")
print(f"  Neg log-likelihood: {result.fun:.4f}")

In [ ]:
# ---- Prediction ----
@njit(parallel=True)
def _mixed_logit_predict(X, beta, halton, n_fixed, n_fixed_params, n_rand_coeffs,
                         n_classes, random_coeff_map, n_halton):
    n_samp = X.shape[0]
    beta_fixed = beta[:n_fixed_params]
    beta_rand_mean = beta[n_fixed_params:n_fixed_params + n_rand_coeffs]
    beta_rand_std_arr = beta[n_fixed_params + n_rand_coeffs:]
    for rc in range(n_rand_coeffs):
        if beta_rand_std_arr[rc] < 0.05:
            beta_rand_std_arr[rc] = 0.05

    probs_all = np.zeros((n_samp, n_classes))
    for s in prange(n_samp):
        probs = np.zeros(n_classes)
        for r in range(n_halton):
            U = np.zeros(n_classes)
            pi = 0
            for c in range(1, n_classes):
                for fi in range(n_fixed):
                    U[c] += X[s, fi] * beta_fixed[pi]
                    pi += 1
            for rc in range(n_rand_coeffs):
                fi_rc = random_coeff_map[rc, 0]
                class_rc = random_coeff_map[rc, 1]
                rb = beta_rand_mean[rc] + beta_rand_std_arr[rc] * halton[r, rc]
                U[class_rc] += X[s, fi_rc] * rb
            max_u = U[0]
            for c in range(1, n_classes):
                if U[c] > max_u:
                    max_u = U[c]
            exp_sum = 0.0
            for c in range(n_classes):
                exp_sum += np.exp(U[c] - max_u)
            for c in range(n_classes):
                probs[c] += np.exp(U[c] - max_u) / exp_sum
        probs_all[s] = probs / n_halton
    return probs_all

In [ ]:
y_prob_rplm = _mixed_logit_predict(X_te, beta_opt, halton_draws, n_fixed, n_fixed_params,
                                    n_rand_coeffs, n_classes, random_coeff_map, N_HALTON)
y_pred_rplm = np.argmax(y_prob_rplm, axis=1)

In [ ]:
rplm_metrics = compute_metrics(y_te, y_pred_rplm, y_prob_rplm, 'RPLM')
rplm_metrics['train_time_s'] = round(rplm_train_time, 2)

In [ ]:
# ---- McFadden Pseudo R² (weighted) ----
null_probs_emp = np.array([(y_tr == c).mean() for c in range(n_classes)])
ll_null = np.sum(sample_weights * np.log(null_probs_emp[y_tr]))
ll_model = -result.fun
mcfadden_r2 = 1 - ll_model / ll_null if ll_null < 0 else 0.0
rplm_metrics['McFadden_R2'] = round(mcfadden_r2, 4)
print(f"\n  McFadden Pseudo R²: {mcfadden_r2:.4f} (paper: 0.21)")

In [ ]:
# ---- Random Parameters Summary ----
print(f"\n  Random Parameter Estimates:")
rc = 0
for feat_name in FORCED_RANDOM_FEATURES:
    for cls_idx, cls_label in [(1, 'Serious'), (2, 'Fatal')]:
        if rc < n_rand_coeffs and all_feat_names[random_coeff_map[rc][0]] == feat_name:
            mean_est = beta_opt[n_fixed_params + rc]
            std_est = beta_opt[n_fixed_params + n_rand_coeffs + rc]
            print(f"    {feat_name:25s}|{cls_label:8s}: mean={mean_est:+.4f}, std={std_est:.4f}")
            rc += 1
    # other_age_55_64 only affects Serious
    if feat_name == 'other_age_55_64':
        pass  # already handled above

In [ ]:
# ---- Fixed Parameters ----
print(f"\n  Fixed Parameter Estimates (top 10 by magnitude):")
fixed_params_list = []
pi = 0
for c in range(1, n_classes):
    cls_label = ['Serious', 'Fatal'][c - 1]
    for fi in range(n_fixed):
        fixed_params_list.append((all_feat_names[fi], cls_label, beta_opt[pi]))
        pi += 1
fixed_params_list.sort(key=lambda x: abs(x[2]), reverse=True)
for feat, cls, val in fixed_params_list[:10]:
    print(f"    {feat:25s}|{cls:8s}: {val:+.6f}")

In [ ]:
# ---- Pseudo-elasticity for speed_limit_ge50 ----
if 'speed_limit_ge50' in all_feat_names:
    sp_idx = all_feat_names.index('speed_limit_ge50')
    X_mod = X_te.copy()
    X_mod[:, sp_idx] = 1
    probs_base = _mixed_logit_predict(X_te, beta_opt, halton_draws, n_fixed, n_fixed_params,
                                       n_rand_coeffs, n_classes, random_coeff_map, N_HALTON)
    probs_mod = _mixed_logit_predict(X_mod, beta_opt, halton_draws, n_fixed, n_fixed_params,
                                      n_rand_coeffs, n_classes, random_coeff_map, N_HALTON)
    mask_changed = X_te[:, sp_idx] == 0
    if mask_changed.sum() > 0:
        elast = (probs_mod[mask_changed] - probs_base[mask_changed]) / np.maximum(probs_base[mask_changed], 1e-10) * 100
        elast_mean = np.mean(elast, axis=0)
        print(f"\n  Pseudo-elasticity (speed_limit_ge50: 0->1):")
        for c, label in enumerate(['Slight', 'Serious', 'Fatal']):
            print(f"    {label:8s}: {elast_mean[c]:+.2f}%")
        rplm_metrics['elasticity_speed_ge50_Slight'] = round(elast_mean[0], 2)
        rplm_metrics['elasticity_speed_ge50_Serious'] = round(elast_mean[1], 2)
        rplm_metrics['elasticity_speed_ge50_Fatal'] = round(elast_mean[2], 2)
else:
    print(f"\n  speed_limit_ge50 not in feature set, skipping elasticity")

In [ ]:
random_param_names = FORCED_RANDOM_FEATURES

In [ ]:
# ============================================================
# 12. RESULTS SUMMARY
# ============================================================
print("\n" + "=" * 70)
print("RESULTS SUMMARY")
print("=" * 70)

In [ ]:
results_df = pd.DataFrame([rsf_metrics, rplm_metrics])
results_df = results_df.transpose()
print(f"\n{results_df.to_string()}")

In [ ]:
results_df.to_csv(os.path.join(OUTPUT_DIR, 'paper_reproduction_results.csv'))

In [ ]:
summary = {
    'dataset': 'UK STATS19 Bicycle Accidents (2016–2018)',
    'n_samples': len(data),
    'n_features': X.shape[1],
    'train_test_split': '70/30',
    'models': {
        'RSF': {
            'hyperparameters': rsf_params,
            'metrics': {k: v for k, v in rsf_metrics.items() if k != 'model'},
            'n_rules_extracted': rules_count,
        },
        'RPLM': {
            'random_parameters': random_param_names,
            'metrics': {k: v for k, v in rplm_metrics.items() if k != 'model'},
            'mcfadden_r2': mcfadden_r2,
        }
    },
    'output_dir': OUTPUT_DIR,
}

In [ ]:
with open(os.path.join(OUTPUT_DIR, 'pipeline_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2, default=str)

In [ ]:
print(f"\n  Results saved to {OUTPUT_DIR}")
print("=" * 70)
print("DONE — Britain Paper Reproduction Complete!")
print("=" * 70)